# Microsoft Agent Framework — Azure OpenAI (Responses API)

यस कोड नमूनामा, तपाईं **Microsoft Agent Framework (MAF)** प्रयोग गरी एक सरल एजेन्ट निर्माण गर्नुहुनेछ जुन **Azure OpenAI** द्वारा समर्थित छ र **Responses API** प्रयोग गर्दछ।

> **स्थानान्तरण नोट:** यो नमूनाले पहिले Semantic Kernel सँग GitHub Models प्रयोग गर्थ्यो। यसलाई Microsoft Agent Framework मा स्थानान्तरण गरिएको छ, र GitHub Models (जे जुन २०२६ मा अवसान हुँदैछ) लाई Azure OpenAI द्वारा प्रतिस्थापन गरिएको छ, जसले Responses API समर्थन गर्दछ। MAF मा रहेको `OpenAIChatClient` Azure OpenAI को स्थिर `/openai/v1/` अन्त बिन्दुमा लक्षित छ र डिफल्टले Responses API प्रयोग गर्दछ।

यस नमूनाको उद्देश्य विभिन्न एजेन्टिक ढाँचा कार्यान्वयन गर्दा पछि थप कोड नमूनाहरूमा लागू गरिने चरणहरू देखाउनु हो।


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## आवश्यक पाइथन प्याकेजहरू आयात गर्नुहोस्


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## उपकरण परिभाषित गर्दै

माइक्रोसफ्ट एजेन्ट फ्रेमवर्कमा, एक **उपकरण** एउटा सामान्य Python कार्य हो जुन `@tool` द्वारा सजाइएको हुन्छ जुन एजेन्टले कल गर्न सक्छ। तल हामी एउटा उपकरण परिभाषित गर्छौं जुन एउटा यादृच्छिक बिदा गन्तव्य फर्काउँछ र अघिल्लो गन्तव्य दोहोरिने छाल्छ। 


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## एजेन्ट सिर्जना गर्दै

यहाँ, हामीले `TravelAgent` नामक एजेन्ट सिर्जना गरिरहेका छौं।

यस उदाहरणमा, हामी धेरै आधारभूत निर्देशनहरू प्रयोग गर्छौं। यी निर्देशनहरू परिवर्तन गरेर एजेन्टको व्यवहार कस्तो हुन्छ हेर्न स्वतन्त्र महसुस गर्नुहोस्।


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## एजेन्ट चलाउँदै हुनुहुन्छ

अब हामी एजेन्टलाई चलाउन सक्छौं। हामीले `AgentSession` सिर्जना गर्छौं ताकि एजेन्टले कुराकानीलाई पालो पालो सम्झन सक्छ, र त्यसपछि दुई `user_inputs` पठाउँछौं। पहिलोले यात्रा माग्छ; दोस्रोले प्रयोगकर्तालाई सुझाव मन नपरेको भन्छ र अर्को माग्छ — एजेन्टले सेसन इतिहास र `get_random_destination` उपकरण प्रयोग गरेर जवाफ दिन्छ।

तपाईं यी सन्देशहरू संशोधन गर्न सक्नुहुन्छ र हेर्न सक्नुहुन्छ कि एजेन्टले फरक प्रतिक्रिया कसरी दिन्छ। जवाफहरू **टोकन-दर-टोकन** प्रवाहित हुन्छन्।


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी सही हुन प्रयास गर्छौं, तर कृपया जानकार हुनुस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धताहरू हुन सक्छन्। मूल दस्तावेज़ यसको मूल भाषामा आधिकारिक स्रोत मानिनुपर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलत बुझाइ वा त्रुटिको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
